# RDDs con PySpark

> **IMPORTANTE**: Este notebook **no funcionará en una cuenta Free Edition de Databricks**. Está pensado para ser ejecutado en un entorno de Jupyter Notebook con PySpark en local. Asegúrate de tener PySpark instalado y configurado correctamente antes de ejecutar el código

En este notebook aprenderemos a:

1. Crear una sesión de Spark (`SparkSession`)
2. Crear RDDs a partir de colecciones de Python y de ficheros externos
3. Ver cómo se reparten los datos de un RDD en particiones
4. Aplicar transformaciones y acciones (word count)
5. Persistir RDDs que se reutilizan varias veces

Para poder ejecutar Spark es necesario disponer de una instalación de Java. Se puede comprobar si Java está instalado en la máquina con el siguiente comando.

In [1]:
!java --version

java 25.0.3 2026-04-21 LTS
Java(TM) SE Runtime Environment (build 25.0.3+9-LTS-195)
Java HotSpot(TM) 64-Bit Server VM (build 25.0.3+9-LTS-195, mixed mode, sharing)


En caso de que el comando anterior falle, será necesario instalar Java en la máquina. Podemos instalar la versión LTS de Java 25 con el siguiente comando si estamos en una máquina con Linux (Debian/Ubuntu; puede requerir `sudo`). Para instalar Java en Windows, tendremos que visitar la siguiente página (https://www.oracle.com/es/java/technologies/downloads)

In [ ]:
!apt install -y openjdk-25-jdk

A continuación, será necesario comprobar si la variable `JAVA_HOME`, que indica a Spark dónde encontrar la JVM para ejecutar su código Scala, está definida. Si no lo está, la siguiente celda dará un error `KeyError`.

In [4]:
import os

os.environ["JAVA_HOME"]

'/usr/lib/jvm/jdk-25.0.3-oracle-x64/'

En caso de que la variable no esté definida, se debe establecer de la siguiente manera.

In [ ]:
# En nuestro ordenador personal, si no esta definida la variable JAVA_HOME, deberemos indicarla
# Para sistemas basados en Debian/Ubuntu, si tenemos instalada la version 25 de Java, seria:
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-25-openjdk-amd64"
# En Windows, la ruta puede ser algo como: "C:\\Program Files\\Java\\jdk-25"
# os.environ["JAVA_HOME"] = "C:\\Program Files\\Java\\jdk-25"
# Si ya esta definida, no es necesario hacer nada
# os.environ["JAVA_HOME"]

A continuación, importaremos PySpark y verificaremos si está correctamente instalado.

In [5]:
import pyspark

pyspark.__version__

'4.2.0'

Ya estamos listos para crear una sesión de Spark y empezar a trabajar con RDDs.

In [6]:
from pyspark.sql import SparkSession

# Crear sesión de Spark
# - builder: permite configurar la sesión antes de crearla
# - appName: nombre de la aplicación (aparece en la interfaz web de Spark)
# - getOrCreate: crea la sesión o reutiliza la existente si ya hay una
# Si no indicamos .master(), en local se usa "local[*]": todos los núcleos del ordenador
spark = SparkSession.builder.appName("Analisis Deportivo con Spark").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 23:09:52 WARN Utils: Your hostname, ThinkPad-T480s, resolves to a loopback address: 127.0.1.1; using 172.17.0.1 instead (on interface docker0)
26/11/11 23:09:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 23:09:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Una vez creado un objeto `SparkSession`, podemos inspeccionarlo. Ahora mismo lo estamos utilizando en modo local, con todos los núcleos de nuestro ordenador -> `local[*]`. El enlace **Spark UI** abre la interfaz web de Spark, donde podemos ver los jobs, stages y tasks que se ejecutan.

In [7]:
spark

### Ejemplo 1 - Crear un RDD desde una colección

Vamos a crear un RDD a partir de una colección de Python. En este caso, una lista con información sobre jugadores y su distancia recorrida en un partido de fútbol.

In [8]:
# Datos simulados: (jugador, distancia recorrida en km)
football_data = [("Jugador1", 10.2), ("Jugador2", 9.8), ("Jugador1", 11.0)]

# Creamos un RDD a partir de la lista
rdd = spark.sparkContext.parallelize(football_data)

# Distancia total por jugador
# reduceByKey es una transformación: es perezosa, todavía no se ejecuta nada
total_distance = rdd.reduceByKey(lambda a, b: a + b)

# collect es una acción: lanza la ejecución y devuelve el resultado al driver
total_distance.collect()

[('Jugador2', 9.8), ('Jugador1', 21.2)]

### Ejemplo 2 - Particiones

Un RDD se divide en **particiones**, que son los trozos de datos que se procesan en paralelo: cada tarea (task) procesa una partición.

Si no indicamos el número de particiones, `parallelize` usa el valor de `defaultParallelism`, que en modo local (`local[*]`) es el número de núcleos de nuestro ordenador. Con `getNumPartitions()` podemos consultar cuántas particiones tiene un RDD.

In [9]:
# Ampliamos los datos del ejemplo 1 con más jugadores
football_data = [("Jugador1", 10.2), ("Jugador2", 9.8), ("Jugador1", 11.0),
                 ("Jugador3", 8.5), ("Jugador2", 10.1), ("Jugador3", 9.0)]

# No indicamos el número de particiones: se usa defaultParallelism
rdd_players = spark.sparkContext.parallelize(football_data)

print(spark.sparkContext.defaultParallelism)  # nº de particiones por defecto
print(rdd_players.getNumPartitions())         # nº de particiones del RDD

8
8


Con `glom()` agrupamos los elementos de cada partición en una lista, de forma que podemos ver cómo se han repartido los datos. Si hay más particiones que elementos, algunas particiones quedarán vacías.

> **Cuidado**: `glom().collect()` trae todos los datos al driver. Úsalo solo con datos pequeños, para aprender o depurar.

In [10]:
# Una lista por partición con los elementos que contiene
rdd_players.glom().collect()

[[],
 [('Jugador1', 10.2)],
 [('Jugador2', 9.8)],
 [('Jugador1', 11.0)],
 [],
 [('Jugador3', 8.5)],
 [('Jugador2', 10.1)],
 [('Jugador3', 9.0)]]

En una transformación **estrecha (narrow)**, como `mapValues`, cada partición del resultado depende de una sola partición de entrada. Los datos se quedan donde estaban y el número de particiones no cambia.

In [11]:
# Pasamos la distancia de km a metros (solo cambia el valor, no la clave)
rdd_meters = rdd_players.mapValues(lambda km: km * 1000)

print(rdd_meters.getNumPartitions())  # mismo nº de particiones que rdd_players
rdd_meters.glom().collect()           # cada elemento sigue en su partición

8


[[],
 [('Jugador1', 10200.0)],
 [('Jugador2', 9800.0)],
 [('Jugador1', 11000.0)],
 [],
 [('Jugador3', 8500.0)],
 [('Jugador2', 10100.0)],
 [('Jugador3', 9000.0)]]

En una transformación **ancha (wide)**, como `reduceByKey`, Spark tiene que redistribuir los datos (shuffle) para que todos los valores de una misma clave acaben en la misma partición. El reparto se hace según el hash de la clave, por eso puede haber particiones vacías.

In [12]:
# Distancia total por jugador: requiere shuffle
rdd_total = rdd_players.reduceByKey(lambda a, b: a + b)

print(rdd_total.getNumPartitions())  # por defecto, mantiene el nº de particiones
rdd_total.glom().collect()           # cada jugador está en una sola partición

8


[[],
 [],
 [],
 [('Jugador2', 19.9)],
 [],
 [('Jugador1', 21.2), ('Jugador3', 17.5)],
 [],
 []]

Las transformaciones anchas aceptan el argumento `numPartitions` para elegir cuántas particiones tendrá el resultado.

In [13]:
# Mismo cálculo, pero el resultado tendrá solo 2 particiones
rdd_total_2 = rdd_players.reduceByKey(lambda a, b: a + b, numPartitions=2)

print(rdd_total_2.getNumPartitions())
rdd_total_2.glom().collect()

2


[[], [('Jugador1', 21.2), ('Jugador2', 19.9), ('Jugador3', 17.5)]]

### Ejemplo 3 - Word count

Veremos una implementación del clásico problema de contar palabras (word count) utilizando RDDs. En este caso, el RDD se crea a partir de un fichero de texto.

El fichero está en la carpeta `data/`, junto a este notebook. Si trabajamos en local con Jupyter Notebook/Lab, podemos acceder a él directamente con una ruta relativa.

In [14]:
file_path = "data/wordcount_data.txt"

In [15]:
# Cada elemento del RDD es una línea del fichero
rdd = spark.sparkContext.textFile(file_path)

Mostramos el número de líneas del fichero

In [16]:
rdd.count()  # Número de líneas

44

Al leer un fichero, el número de particiones depende de su tamaño (en local, como mínimo 2 por defecto).

In [17]:
rdd.getNumPartitions()

2

In [18]:
# Transformaciones (perezosas): todavía no se ejecuta nada
rdd_word_count = (rdd
    .flatMap(lambda line: line.split())  # cada línea -> varias palabras
    .map(lambda word: (word, 1))         # cada palabra -> (palabra, 1)
    .reduceByKey(lambda x, y: x + y))    # suma los 1 de cada palabra (shuffle)

Guardamos el resultado con la acción `saveAsTextFile`, que lanza la ejecución. Spark crea una **carpeta** con un fichero `part-XXXXX` por cada partición del RDD.

In [19]:
# Si el directorio ya existe, da error. Podemos borrarlo desde la terminal con: rm -rf output_wordcount/
rdd_word_count.saveAsTextFile("output_wordcount")

In [20]:
# Las 10 palabras más frecuentes: ordenamos por el contador (x[1]) de mayor a menor
rdd_word_count.takeOrdered(10, key=lambda x: -x[1])

[('the', 38),
 ('a', 28),
 ('of', 25),
 ('word', 24),
 ('and', 23),
 ('words', 21),
 ('is', 19),
 ('to', 18),
 ('count', 11),
 ('in', 11)]

### Ejemplo 4 - Persistencia

Por la evaluación perezosa, cada acción vuelve a calcular toda la cadena de transformaciones desde el principio. En el ejemplo 3, `rdd_word_count` se ha calculado dos veces: una con `saveAsTextFile` y otra con `takeOrdered`, leyendo el fichero y repitiendo `flatMap`, `map` y `reduceByKey` en cada una.

Si un RDD se va a reutilizar varias veces, conviene persistirlo con `cache()` o `persist()`. Por defecto, un RDD no está persistido.

In [21]:
print(rdd_word_count.is_cached)
print(rdd_word_count.getStorageLevel())

False
Serialized 1x Replicated


In [22]:
rdd_word_count.cache()  # equivale a persist(StorageLevel.MEMORY_ONLY)

print(rdd_word_count.is_cached)
print(rdd_word_count.getStorageLevel())

True
Memory Serialized 1x Replicated


`cache()` también es perezoso: los datos se guardan en memoria la primera vez que se ejecuta una acción. Las siguientes acciones reutilizan los datos en memoria sin volver a calcular el RDD.

Mientras el RDD está persistido, podemos verlo en la pestaña **Storage** de la interfaz web de Spark (http://localhost:4040).

In [23]:
rdd_word_count.count()  # 1ª acción: calcula el RDD y lo guarda en memoria

381

In [24]:
rdd_word_count.takeOrdered(5, key=lambda x: -x[1])  # 2ª acción: reutiliza los datos en memoria

[('the', 38), ('a', 28), ('of', 25), ('word', 24), ('and', 23)]

Cuando ya no lo necesitemos, liberamos la memoria con `unpersist()`.

Si los datos no caben en memoria, podemos elegir otro nivel de almacenamiento con `persist()`. Por ejemplo, con `MEMORY_AND_DISK` las particiones que no quepan en memoria se guardan en disco.

In [25]:
from pyspark import StorageLevel

rdd_word_count.unpersist()  # libera la memoria

rdd_word_count.persist(StorageLevel.MEMORY_AND_DISK)  # si no cabe en memoria, usa disco
print(rdd_word_count.getStorageLevel())
print(rdd_word_count.count())

rdd_word_count.unpersist();  # el ; evita que Jupyter muestre el RDD devuelto

Disk Memory Serialized 1x Replicated
381


Finalmente, podemos cerrar la sesión de Spark cuando ya no la necesitemos. Si lo hacemos, habrá que ejecutar todo el notebook de nuevo para volver a crearla.

In [26]:
spark.stop()